In [1]:
!pip install --pre -U langchain langchain_core langchain-openai langsmith
!pip install langchain-huggingface langchain-qdrant qdrant-client transformers accelerate bitsandbytes

In [2]:
import torch
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

from google.colab import userdata
import os

In [3]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "skripsi"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["QDRANT_API_KEY"] = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_URL"] = userdata.get('QDRANT_URL')

QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
QDRANT_URL = userdata.get('QDRANT_URL')
LANGSMITH_TRACING=True
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_PROJECT="skripsi"
LANGSMITH_API_KEY=userdata.get('LANGSMITH_API_KEY')
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
EMBEDDING_MODEL = "google/embeddinggemma-300m"
COLLECTION_NAME = "medical_reports"
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

In [12]:
# 1. Setup Qdrant Client (Koneksi ke Vector DB)
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# 2. Setup Embedding (Harus SAMA dengan model yang dipakai saat Indexing)
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cuda'}
)

# 3. Hubungkan ke Collection 'medical_reports'
vectorstore = QdrantVectorStore(
    client=client,
    collection_name="medical_reports",
    embedding=embedding_model,
    content_payload_key="full_text",
    # metadata_payload_key=["report_title", "file", "page_number", "chunk_index"]
)

# Setup Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [13]:
# 4. Load LLM: Qwen/Qwen3-4B-Instruct-2507
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load model dengan kuantisasi 4-bit agar hemat RAM
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    load_in_4bit=True, # Butuh library 'bitsandbytes'
    trust_remote_code=True
)

# Buat Pipeline Hugging Face
text_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1, # Temperature rendah untuk data medis (faktual)
    repetition_penalty=1.1
)

llm = HuggingFacePipeline(pipeline=text_pipe)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


In [19]:
# 5. Setup Prompt (Disesuaikan untuk Bahasa Indonesia/Medis)
template = """<|im_start|>system
Anda adalah asisten AI yang efisien untuk analisis laporan medis.
Jawablah pertanyaan HANYA berdasarkan konteks yang diberikan.

ATURAN PENTING:
1. Jawab dengan SINGKAT, PADAT, dan LANGSUNG pada intinya.
2. DILARANG mengulang-ulang kalimat atau mengulang pertanyaan user.
3. Jangan menambahkan basa-basi seperti "Berdasarkan konteks yang diberikan...". Langsung berikan jawabannya.
4. Jika informasi tidak ada di konteks, katakan "Informasi tidak ditemukan".
<|im_end|>
<|im_start|>user
KONTEKS:
{context}

PERTANYAAN:
{question}
<|im_end|>
<|im_start|>assistant
"""
prompt = ChatPromptTemplate.from_template(template)

In [20]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [21]:
anthropometric_assessment = "Jenis Kelamin : Laki-laki\n• Tanggal lahir : 27 Maret 2021\n• Usia : 1 tahun 1 bulan\nPB = 65 cm. \nBBI/U menurut PB \nadalah 8,8 kg."
medical_diagnosis = "Diagnosis : Severe Obstructive Hydrocephalus Post Repair VP \nShunt + Labiopalatoschisis + CP + Pneumonia"
question = "Bagaimana status gizi pada pasien ini?"

In [22]:
full_question = anthropometric_assessment + "\n\n" + medical_diagnosis + "\n\n" + question
response = rag_chain.invoke(full_question)

In [23]:
response

'Human: <|im_start|>system\nAnda adalah asisten AI yang efisien untuk analisis laporan medis. \nJawablah pertanyaan HANYA berdasarkan konteks yang diberikan.\n\nATURAN PENTING:\n1. Jawab dengan SINGKAT, PADAT, dan LANGSUNG pada intinya.\n2. DILARANG mengulang-ulang kalimat atau mengulang pertanyaan user.\n3. Jangan menambahkan basa-basi seperti "Berdasarkan konteks yang diberikan...". Langsung berikan jawabannya.\n4. Jika informasi tidak ada di konteks, katakan "Informasi tidak ditemukan".\n<|im_end|>\n<|im_start|>user\nKONTEKS:\n\n    Anthropometry: Jenis Kelamin : Laki-laki\n• Tanggal lahir : 27 Maret 2021\n• Usia : 1 tahun 1 bulan\nPB = 65 cm. \nBBI/U menurut PB \nadalah 8,8 kg.\n    Diagnosis: Diagnosis : Severe Obstructive Hydrocephalus Post Repair VP \nShunt + Labiopalatoschisis + CP + Pneumonia\n    Context: 2.4.2 Diagnosis Gizi\n• NI 2.9 – Keterbatasan Penerimaan Makanan berkaitan dengan pasien\ndipuasakan saat sebelum dan setelah operasi, ditandai dengan asupan energi\n(52%), 